# CMPE 460 Mendeley Dataset Inspection

**Member 2**

This notebook inspects the baseline CSV from version 2 of the Multi-Class Web Attack Dataset. The published description reports 25,736 requests and six classes. The audit measures the downloaded file before preprocessing decisions are made.

Download `baseline_dataset.csv` and place it in `data/mendeley/` inside the repository. Run the cells from top to bottom. Only aggregate statistics are displayed.

[Dataset page and download](https://data.mendeley.com/datasets/zhzd5j9ffy/2)

Popović, M., and Vuletić, P. (2026). Multi-Class Web Attack Dataset with Iterative LLM-Generated Adversarial Payloads. Mendeley Data, V2. DOI 10.17632/zhzd5j9ffy.2. Licensed under CC BY 4.0.


## 1. Load the Baseline CSV

The CSV is read as strings to preserve literal payload values such as `NA`, `null`, and URL-encoded characters. Empty fields remain empty strings. The SHA-256 identifies the exact downloaded file.

If pandas is missing, install it once using `%pip install pandas` in a separate code cell.


In [1]:
import hashlib
from pathlib import Path

import pandas as pd
from IPython.display import display

CSV_PATH = Path("data/mendeley/baseline_dataset.csv")
EXPECTED_ROWS = 25736

if not CSV_PATH.is_file():
    raise FileNotFoundError(
        "Download baseline_dataset.csv from "
        "https://data.mendeley.com/datasets/zhzd5j9ffy/2 "
        f"and place it at {CSV_PATH.resolve()}."
    )

raw = pd.read_csv(
    CSV_PATH,
    dtype="string",
    keep_default_na=False,
    encoding="utf-8-sig",
)
if raw.empty:
    raise ValueError("The CSV contains no data rows.")

file_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()

print(f"File: {CSV_PATH.name}")
print(f"SHA-256: {file_sha256}")
print(f"Rows: {len(raw):,}")
print(f"Columns: {len(raw.columns):,}")
display(pd.DataFrame({"column": raw.columns.tolist()}))


File: baseline_dataset.csv
SHA-256: d3b3509555924de6e1d37522806f2debd97ea6659e8254a95ad98d19c28a24d6
Rows: 25,736
Columns: 60


,column
0,method
1,url
2,payload
3,req_is_post
4,url_len
5,body_len
6,url_param_count
7,url_max_param_len
8,url_avg_param_len
9,body_param_count


## 2. Identify Text Fields and Labels

The input fields are HTTP method, URL, and payload. Column matching ignores capitalization and surrounding spaces. If the CSV uses other names, update `SOURCE_COLUMNS` using the column list above.

The numeric label mapping comes from the version 2 dataset description. The source includes 56 extracted features. The candidate text below is constructed from the three HTTP fields.

Field markers make the candidate representation explicit. Characters, case, punctuation, and URL encoding are preserved. Review the length statistics before deciding how to encode the final model input.


In [2]:
SOURCE_COLUMNS = {
    "method": "method",
    "url": "url",
    "payload": "payload",
    "label": "label",
}

LABEL_NAMES = {
    0: "normal",
    1: "xss",
    2: "sqli",
    3: "cmdi",
    4: "lfi",
    5: "ssrf",
}
TEXT_COLUMNS = ["method", "url", "payload"]

normalized_columns = {
    str(column).strip().casefold(): column
    for column in raw.columns
}
if len(normalized_columns) != len(raw.columns):
    raise ValueError("Column names become ambiguous after ignoring case and spaces.")

missing_columns = [
    source
    for source in SOURCE_COLUMNS.values()
    if source.strip().casefold() not in normalized_columns
]
if missing_columns:
    raise ValueError(
        f"Missing configured columns: {missing_columns}. "
        "Update SOURCE_COLUMNS using the column list printed above."
    )

actual_columns = {
    field: normalized_columns[source.strip().casefold()]
    for field, source in SOURCE_COLUMNS.items()
}
fields = raw[list(actual_columns.values())].copy()
fields.columns = list(actual_columns)

parsed_labels = pd.to_numeric(fields["label"], errors="coerce")
valid_labels = parsed_labels.isin(LABEL_NAMES)

candidate_text = (
    "[METHOD] " + fields["method"]
    + "\n[URL] " + fields["url"]
    + "\n[PAYLOAD] " + fields["payload"]
)

display(pd.DataFrame({
    "input_field": list(actual_columns),
    "csv_column": list(actual_columns.values()),
}))
display(pd.Series(LABEL_NAMES, name="class").rename_axis("label_id").to_frame())


,input_field,csv_column
0,method,method
1,url,url
2,payload,payload
3,label,label


,class
label_id,
0,normal
1,xss
2,sqli
3,cmdi
4,lfi
5,ssrf


## 3. Audit Empty Fields and Repeated Inputs

An empty input has blank method, URL, and payload fields. Empty payloads are counted individually because a request can have a URL and method with an empty payload.

The audit counts repeated complete rows, repeated non-empty candidate inputs, and candidate inputs carrying multiple recognized labels. The original rows are retained during inspection.


In [3]:
empty_fields = pd.DataFrame({
    column: fields[column].str.strip().eq("")
    for column in TEXT_COLUMNS
})
empty_inputs = empty_fields.all(axis=1)
missing_labels = fields["label"].str.strip().eq("")

input_duplicate_rows = int(candidate_text[~empty_inputs].duplicated().sum())

label_audit = pd.DataFrame({
    "input_text": candidate_text,
    "label": parsed_labels,
})
label_audit = label_audit.loc[valid_labels & ~empty_inputs]
labels_per_input = label_audit.groupby("input_text")["label"].nunique()
conflicting_input_groups = int(labels_per_input.gt(1).sum())

quality = pd.Series({
    "rows": len(raw),
    "columns": len(raw.columns),
    "repeated_complete_rows": int(raw.duplicated().sum()),
    "empty_input_rows": int(empty_inputs.sum()),
    "missing_labels": int(missing_labels.sum()),
    "invalid_labels_including_missing": int((~valid_labels).sum()),
    "repeated_nonempty_input_rows": input_duplicate_rows,
    "inputs_with_conflicting_labels": conflicting_input_groups,
}, name="count")
display(quality.to_frame())
display(empty_fields.sum().rename("empty_rows").rename_axis("field").to_frame())


,count
rows,25736
columns,60
repeated_complete_rows,0
empty_input_rows,0
missing_labels,0
invalid_labels_including_missing,0
repeated_nonempty_input_rows,0
inputs_with_conflicting_labels,0


,empty_rows
field,
method,0
url,0
payload,21735


## 4. Inspect Class Counts and Candidate Lengths

Class counts describe the original rows with recognized labels. Shares use the total CSV row count as the denominator. Length statistics describe non-empty candidate inputs.

These full-file statistics support the data audit. In preprocessing, partition the data after duplicate checks and select the maximum sequence length using the training split.


In [4]:
class_counts = (
    parsed_labels[valid_labels]
    .astype("int64")
    .value_counts()
    .reindex(list(LABEL_NAMES), fill_value=0)
    .sort_index()
)
class_distribution = pd.DataFrame({
    "class": pd.Series(LABEL_NAMES),
    "rows": class_counts,
    "share_pct": 100 * class_counts / len(raw),
}).rename_axis("label_id")
display(class_distribution.round(2))

lengths = candidate_text[~empty_inputs].str.len()
length_summary = pd.Series({
    "nonempty_inputs": len(lengths),
    "min_chars": lengths.min(),
    "median_chars": lengths.median(),
    "p95_chars": lengths.quantile(0.95),
    "p99_chars": lengths.quantile(0.99),
    "max_chars": lengths.max(),
    "inputs_above_256_chars": int(lengths.gt(256).sum()),
}, name="value", dtype="Float64")
display(length_summary.round(1).to_frame())

field_lengths = fields[TEXT_COLUMNS].apply(lambda column: column.str.len())
display(field_lengths.quantile([0.5, 0.95, 0.99]).rename_axis("quantile").round(1))


,class,rows,share_pct
label_id,,,
0,normal,12000,46.63
1,xss,3057,11.88
2,sqli,3216,12.50
3,cmdi,2568,9.98
4,lfi,3000,11.66
5,ssrf,1895,7.36


,value
nonempty_inputs,25736.0
min_chars,45.0
median_chars,134.0
p95_chars,284.0
p99_chars,472.7
max_chars,7170.0
inputs_above_256_chars,1757.0


,method,url,payload
quantile,,,
0.50,3.0,88.0,0.0
0.95,4.0,218.0,111.2
0.99,4.0,392.0,215.0


## 5. Initial Observations

The final cell summarizes the measured counts and the remaining preprocessing decisions. Review its output before creating train, validation, and test partitions. Investigate repeated inputs and label conflicts before assigning rows to splits.


In [5]:
present_classes = int(class_counts.gt(0).sum())

print(f"Published baseline size: {EXPECTED_ROWS:,} rows.")
print(f"Observed size: {len(raw):,} rows.")
print(f"Class coverage: {present_classes}/{len(LABEL_NAMES)} classes.")
print(f"Empty inputs: {int(empty_inputs.sum()):,}.")
print(f"Missing labels: {int(missing_labels.sum()):,}.")
print(f"Invalid labels including missing values: {int((~valid_labels).sum()):,}.")
print(f"Repeated non-empty input rows: {input_duplicate_rows:,}.")
print(f"Inputs with conflicting labels: {conflicting_input_groups:,}.")

if len(lengths):
    print(f"Median candidate length: {lengths.median():,.1f} characters.")
    print(f"95th-percentile candidate length: {lengths.quantile(0.95):,.1f} characters.")
else:
    print("The file has no non-empty candidate inputs.")

if len(raw) != EXPECTED_ROWS:
    print("Check the dataset version and downloaded filename against the source page.")

if (
    input_duplicate_rows
    or conflicting_input_groups
    or empty_inputs.any()
    or (~valid_labels).any()
):
    print("Document a data-cleaning policy before partitioning the dataset.")
else:
    print("The audit found zero empty inputs, invalid labels, input repeats, and label conflicts.")

print("Next step: create reproducible splits and fit the character vocabulary on training data.")


Published baseline size: 25,736 rows.
Observed size: 25,736 rows.
Class coverage: 6/6 classes.
Empty inputs: 0.
Missing labels: 0.
Invalid labels including missing values: 0.
Repeated non-empty input rows: 0.
Inputs with conflicting labels: 0.
Median candidate length: 134.0 characters.
95th-percentile candidate length: 284.0 characters.
The audit found zero empty inputs, invalid labels, input repeats, and label conflicts.
Next step: create reproducible splits and fit the character vocabulary on training data.
